# Streaming Statistics — Files per Batch

Runs an isolated Auto Loader stream (separate checkpoint, schema location, and target table — not the main `netflix_titles_stream` pipeline) with `maxFilesPerTrigger` set low enough to produce multiple batches, then inspects batch-level progress via `query.recentProgress`.

## 1. Configuration

In [0]:
%run ./lab3_00_config

In [0]:
pipeline_name = "lab3_statistics"

source_path = f"{storage_root}/ingestion/netflix_stream/"
stats_checkpoint_path = f"{storage_root}/checkpoints/{pipeline_name}/"
stats_schema_location = f"{storage_root}/schema_location/{pipeline_name}/"
stats_table = f"{catalog}.{bronze_schema}.netflix_titles_stream_stats"

print("Source path:", source_path)
print("Stats checkpoint:", stats_checkpoint_path)
print("Stats schema location:", stats_schema_location)
print("Stats table:", stats_table)

## 2. Isolated run with `maxFilesPerTrigger` — produce multiple batches

In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date

df_stats_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", stats_schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", "100")
    .load(source_path)
)

df_stats_stream = (
    df_stats_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

stats_query = (
    df_stats_stream.writeStream
    .option("checkpointLocation", stats_checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(stats_table)
)

stats_query.awaitTermination()
print("Isolated statistics run completed")

## 3. Inspect batch-level progress

In [0]:
progress = stats_query.recentProgress

print(f"Number of batches: {len(progress)}")
print()

for p in progress:
    source_info = p["sources"][0] if p.get("sources") else {}
    num_input_rows = source_info.get("numInputRows")
    duration = p.get("durationMs", {}).get("triggerExecution")
    print(f"Batch {p['batchId']}: numInputRows={num_input_rows}, durationMs={duration}")

## 4. Verify total row count

In [0]:
total_rows = spark.table(stats_table).count()
print("Total rows in isolated stats table:", total_rows)

## 5. Trigger Type Comparison — availableNow vs once

Compares two trigger modes on the same isolated source: `availableNow` (recommended, processes all currently available files then stops) and `once` (deprecated alias with similar semantics).

In [0]:
dbutils.fs.rm(stats_checkpoint_path, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {stats_table}")

print("Isolated stats environment reset for trigger experiment")

### Run 1 — trigger(availableNow=True)

In [0]:
df_stats_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", stats_schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", "100")
    .load(source_path)
)

df_stats_stream = (
    df_stats_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

query_available_now = (
    df_stats_stream.writeStream
    .option("checkpointLocation", stats_checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(stats_table)
)

query_available_now.awaitTermination()

progress_available_now = query_available_now.recentProgress
print(f"[availableNow] Number of batches: {len(progress_available_now)}")
print(f"[availableNow] Total rows loaded: {spark.table(stats_table).count()}")

In [0]:
dbutils.fs.rm(stats_checkpoint_path, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {stats_table}")

print("Isolated stats environment reset for once-trigger run")

### Run 2 — trigger(once=True)

In [0]:
df_stats_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", stats_schema_location)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", "100")
    .load(source_path)
)

df_stats_stream = (
    df_stats_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_load_date", current_date())
)

query_once = (
    df_stats_stream.writeStream
    .option("checkpointLocation", stats_checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(once=True)
    .toTable(stats_table)
)

query_once.awaitTermination()

progress_once = query_once.recentProgress
print(f"[once] Number of batches: {len(progress_once)}")
print(f"[once] Total rows loaded: {spark.table(stats_table).count()}")

## Summary — trigger comparison

Both trigger modes loaded the same 8810 rows, but with very different execution behavior:

- `availableNow=True` respected `maxFilesPerTrigger` and split the work into 11 manageable batches.
- `once=True` ignored `maxFilesPerTrigger` and processed everything in a single batch.

This is exactly why Databricks recommends `availableNow` over the deprecated `once` — it gives more control, better batch-level observability, and more predictable resource usage.